# 03 - Transfer Governance Before Export

This notebook demonstrates destination-aware authorization. The same source table produces different decisions depending on destination system, destination jurisdiction, and consumer jurisdiction.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

client = get_client()
mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
print(f"Metatate examples mode: {mode}")


## Approved destination with required controls


In [ ]:
table_name = "ACMECLOUD_DEMO.PUBLIC.CUSTOMERS"
salesforce = client.authorize_use(
    table_name,
    operation="export",
    intended_use="external_sharing",
    actor_role="DATA_ENGINEER",
    columns=["CUSTOMER_ID", "CUSTOMER_NAME", "EMAIL", "ACCOUNT_STATUS"],
    destination={"system": "SALESFORCE", "jurisdiction": "US"},
    consumer_jurisdiction="EU",
)
print(json.dumps(salesforce["data"], indent=2))


## Explain the conditional decision


In [ ]:
decision_id = salesforce["data"]["decision_id"]
explanation = client.explain_why(decision_id=decision_id)
print(json.dumps(explanation["data"], indent=2))


## Prohibited destination


In [ ]:
ads_platform = client.authorize_use(
    table_name,
    operation="export",
    intended_use="external_sharing",
    actor_role="DATA_ENGINEER",
    columns=["CUSTOMER_ID", "CUSTOMER_NAME", "EMAIL"],
    destination={"system": "ADS_PLATFORM", "jurisdiction": "US"},
    consumer_jurisdiction="US",
)
print(json.dumps(ads_platform["data"], indent=2))


A useful agent should not simply generate an export query. It should ask the data's decision layer whether the transfer is allowed for the actual destination.
